# Fig 8: TBT Heat Map Difference (300W vs 200W)

**Requires sudo:** `sudo nvidia-smi -i 1,2,3 -pl 200` to set the 200W power cap before running the collect cell.  
**Output:** `paper/figures/section3/output/decode_grid_heatmap_diff.{pdf,png}`

### Call order
1. Collect 300W decode-grid data and summary (see Fig 4 notebook): `profiling/launch_decode_grid.sh` → `scripts/plot_decode_grid.py` → `output/300W/decode_grid_cell_summary.csv`
2. Collect 200W decode-grid data at 200W cap and produce `output/200W/decode_grid_cell_summary.csv`
3. `scripts/plot_decode_grid_diff.py`

Steps 1–2 collect raw data and only need to be run once. If data is available from another machine, copy `decode_grid_cell_summary.csv` into both `paper/figures/section3/output/300W/` and `paper/figures/section3/output/200W/` and skip the collect cell below.

In [ ]:
import subprocess
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd()] + list(Path.cwd().parents)
    if (p / ".conserve_root").exists()
)


def run(cmd):
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, end="")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {' '.join(str(c) for c in cmd)}")

In [ ]:
# Steps 1-2: collect decode-grid data at both power levels. Skip if summaries already exist.
# Note: 300W data is also required by Fig 4 — see that notebook to collect it.
# For 200W: set GPU power cap before running (requires sudo):
#   sudo nvidia-smi -i <GPU_ID> -pl 200
# Then run launch_decode_grid.sh, copy output to output/200W/decode_grid_data/,
# and run plot_decode_grid.py with DATA/OUT pointing to output/200W/.
summary_300w = REPO_ROOT / "paper/figures/section3/output/300W/decode_grid_cell_summary.csv"
if not summary_300w.exists():
    run(["bash", str(REPO_ROOT / "profiling/launch_decode_grid.sh")])
    run(["python", str(REPO_ROOT / "paper/figures/section3/scripts/plot_decode_grid.py")])

In [ ]:
# Step 3: plot diff (requires both output/300W/ and output/200W/ summaries)
%matplotlib inline
%run ../scripts/plot_decode_grid_diff.py

In [ ]:
from IPython.display import Image

Image(str(REPO_ROOT / "paper/figures/section3/output/decode_grid_heatmap_diff.png"))